In [0]:
-- Before we do the silver transformation, we need to verify the bronze layer data types:

DESCRIBE databricks_cnc_machine_project.cnc_machine_bronze_layer

-- In the next table we show the bronze layer data types (obtained using the results of the prevous query) and we decide what data type we want for the silver table:

/* Table with Data Types:

|Column                 |Bronze Data Type|Silver Data Type|
|-----------------------|----------------|----------------|
|UDI                    |bigint          |INT             |
|Product ID             |string          |STRING          |
|Type                   |string          |STRING          |
|Air temperature [K]    |double          |DOUBLE          |
|Process temperature [K]|double          |DOUBLE          |
|Rotational speed [rpm] |bigint          |INT             |
|Torque [Nm]            |double          |DOUBLE          |
|Tool wear [min]        |bigint          |INT             |
|Machine failure        |bigint          |BOOLEAN         |
|TWF                    |bigint          |BOOLEAN         |
|HDF                    |bigint          |BOOLEAN         |
|PWF                    |bigint          |BOOLEAN         |
|OSF                    |bigint          |BOOLEAN         |
|RNF                    |bigint          |BOOLEAN         |
|_load_timestamp        |timestamp       |N.A             |
*/

-- For columns like Product ID and Type, we maintain the string data type. Others, like UDI and Tool wear, we would like to change the data type from BIGINT to INT, optimizing storage. In the case of Machine failure and Failure Modes columns like TWF, HDF and so on, we want to change the data type from BIGINT to BOOLEAN.

-- We're also going to check if there are rows with ilogic or bad information in the bronze layer, running queries to identify them.

-- There are at least two scenarios: a machine failure is identified without a failure mode, or a failure mode is identified without a machine failure. This could happen for many reasons, for example, sensor failures or problems before or during the data ingestion process.


-- In this query we verify if there are rows with a machine failure without an identification for a failure mode:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `Machine failure` = 1 AND (`TWF` = 0 AND `HDF` = 0 AND `PWF` = 0 AND `OSF` = 0 AND `RNF` = 0)

-- For this dataset, there are 9 rows with this problem.

-- With the next query, we verify if there are rows that exhibit a failure mode without a machine failure:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `Machine failure` = 0 AND (`TWF` = 1 OR `HDF` = 1 OR `PWF` = 1 OR `OSF` = 1 OR `RNF` = 1)

-- There are 18 rows with the problem identified above. But, all 18 rows are related to RNF or Random Failure, which could lead us to think that could be a problem with the data upstream. For this reason, we run the next query to evaluate how many rows exhibit only random failures:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `RNF` = 1

-- There are in total 19 rows with random failure, almost indentical with those which doesn't exihibit machine failure. 
-- In this case, we decided to retain the data in the silver table for the 18 records of random failures that don't indicate a machine failure; this data may have been generated because a random failure doesn't necessarily mean that the machining process must end abruptly (for example, an alarm was triggered by a sensor, but the process was completed).

-- However, in the case of the 9 records that indicate a machine failure without specifying the failure mode, the decision has been made to remove them from the silver table and place them in a separate quarantine table, as it is highly likely that these are erroneous records or that they deserve further analysis to provide an alternative explanation (since additional failure modes may need to be identified).

-- Table created to quarantine records displaying a machine failure without a failure mode:

CREATE OR REPLACE TABLE databricks_cnc_machine_project.cnc_machine_silver_quarantine AS

(SELECT
  ROW_NUMBER() OVER (ORDER BY UDI) AS quarantine_id,
  UDI::INT AS original_udi,
  `Product ID` AS product_id,
  `Type` AS product_type,
  `Air temperature [K]` AS air_temperature_k,
  `Process temperature [K]` AS process_temperature_k,
  `Rotational speed [rpm]`::INT AS rotational_speed_rpm,
  `Torque [Nm]` AS torque_nm,
  `Tool wear [min]`::INT AS tool_wear_min,
  `Machine failure`::BOOLEAN AS machine_failure,
  TWF::BOOLEAN AS tool_wear_failure,
  HDF::BOOLEAN AS heat_dissipation_failure,
  PWF::BOOLEAN AS power_failure,
  OSF::BOOLEAN AS overstrain_failure,
  RNF::BOOLEAN AS random_failure,
  'Machine failure without a failure mode' AS quarantine_reason,
  current_timestamp() AS _quarantine_timestamp
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `Machine failure` = 1 AND (`TWF` = 0 AND `HDF` = 0 AND `PWF` = 0 AND `OSF` = 0 AND `RNF` = 0))

-- The quarantine table has been created. Now, we can execute a general query on it:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_silver_quarantine




In [0]:
-- Next step, we run a transformation query for fixing column tags and data types. This helps with the workflow for analytics and visualization in the gold layer and dashboards. 
-- Also, taking into account the data types from the bronze layer, we're replacing failure mode numbers 0's and 1's with boolean logic, optimizing storage and performance (0=no failure during the CNC operation; 1=machine failure and specific failure mode). We're also taking into account bigint to int data type transformation. Double data types are maintained for better accuracy and precision in engineering calculations.
-- Lastly, we're calculating power consumption and adding a column with the timestamp of the transformation from bronze to silver layer for traceability.


-- VERIFICAR ESTE PASO LOS DATA TYPES Y EL WHERE
CREATE OR REPLACE TABLE databricks_cnc_machine_project.cnc_machine_silver_layer AS

(SELECT 
    row_number() OVER (ORDER BY UDI) AS silver_id,

    -- Changing column tags with optimized and legible description.
    UDI::INT AS original_udi,
    `Product ID` AS product_id,
    `Type` AS product_type,
    `Air temperature [K]`::DOUBLE AS air_temperature_k,
    `Process temperature [K]`::DOUBLE AS process_temperature_k,
    `Rotational speed [rpm]`::INT AS rotational_speed_rpm,
    `Torque [Nm]`::DOUBLE AS torque_nm,
    `Tool wear [min]`::INT AS tool_wear_min,

    -- Changing one or zero numbers by boolean logic:
    `Machine failure`::BOOLEAN AS has_machine_failure,
    TWF::BOOLEAN AS tool_wear_failure,
    HDF::BOOLEAN AS heat_dissipation_failure,
    PWF::BOOLEAN AS power_failure,
    OSF::BOOLEAN AS overstrain_failure,
    RNF::BOOLEAN AS random_failure,

    -- Power calculation (in Watts) for comparison and analytics in gold layer and dashboards:
    ROUND((`Torque [Nm]` * `Rotational speed [rpm]`) / 9.548, 2)::DOUBLE AS power_watts,

    -- Silver layer transformation timestamp:
    current_timestamp() AS _transformed_timestamp
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE NOT `Machine failure` = 1 AND (`TWF` = 0 AND `HDF` = 0 AND `PWF` = 0 AND `OSF` = 0 AND `RNF` = 0))
    

